In [ ]:
import os
from dotenv import load_dotenv

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
)
from langchain_core.output_parsers import StrOutputParser

from langsmith import traceable


# --------------------------------------------------
# Environment
# --------------------------------------------------

load_dotenv()

os.environ["LANGCHAIN_PROJECT"] = "RAG Chatbot"

# --------------------------------------------------
# PDF
# --------------------------------------------------

PDF_PATH = "Conor_McGregor_Overview.pdf"


# --------------------------------------------------
# Load PDF
# --------------------------------------------------

@traceable(name="load_pdf")
def load_pdf(path: str):
    loader = PyPDFLoader(path)
    return loader.load()


# --------------------------------------------------
# Split Documents
# --------------------------------------------------

@traceable(name="split_documents")
def split_documents(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150,
    )

    return splitter.split_documents(docs)


# --------------------------------------------------
# Build Vector Store
# --------------------------------------------------

@traceable(name="build_vectorstore")
def build_vectorstore(splits):

    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001"
    )

    vector_db = Chroma.from_documents(
        documents=splits,
        embedding=embeddings,
        persist_directory="./chroma_db",
    )

    return vector_db


# --------------------------------------------------
# Pipeline
# --------------------------------------------------

@traceable(name="setup_pipeline")
def setup_pipeline(pdf_path):

    docs = load_pdf(pdf_path)

    splits = split_documents(docs)

    vector_db = build_vectorstore(splits)

    return vector_db


# Build everything once
vector_db = setup_pipeline(PDF_PATH)

retriever = vector_db.as_retriever(
    search_kwargs={"k": 3}
)

# --------------------------------------------------
# LLM
# --------------------------------------------------

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
)

# --------------------------------------------------
# Prompt
# --------------------------------------------------

prompt = ChatPromptTemplate.from_template(
"""
You are a helpful assistant.

Answer ONLY using the provided context.

If the answer is not available in the context, reply exactly:

"I couldn't find that information in the PDF."

Context:
{context}

Question:
{question}

Answer:
"""
)

# --------------------------------------------------
# Format Docs
# --------------------------------------------------

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# --------------------------------------------------
# Parallel Chain
# --------------------------------------------------

parallel_chain = RunnableParallel(
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
)

# --------------------------------------------------
# RAG Chain
# --------------------------------------------------

rag_chain = (parallel_chain | prompt | llm | StrOutputParser())

# --------------------------------------------------
# Chat
# --------------------------------------------------

print("=" * 60)
print("Simple PDF RAG")
print("=" * 60)

while True:

    question = input("\nQuestion: ")

    if question.lower() == "exit":
        break

    answer = rag_chain.invoke(question)

    print("\nAnswer:\n")
    print(answer)

HI
